# nb12 — Pretrained WakeHuBERT featurizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TigreGotico/wakeforge/blob/dev/notebooks/nb12_wakehubert.ipynb)

This notebook trains a wake-word detector for a phrase you type, on top of **WakeHuBERT**:
a 0.64M-parameter streaming speech feature extractor distilled from HuBERT-base. It turns
16 kHz audio into 128-dimensional features at 50 frames per second, and each frame is
computed from the 2.5 s of audio ending with its own 20 ms hop, so it runs on a live stream.
wakeforge downloads it from the Hugging Face
model repository [`TigreGotico/wakehubert-tiny`](https://huggingface.co/TigreGotico/wakehubert-tiny)
when you select `featurizer_type="wakehubert"` (or `"wakehubert-int8"`).

Steps:

1. Install wakeforge.
2. Generate synthetic positives for the wake word with wakeforge's datagen, and download
   negatives and augmentation audio the way the quickstart does.
3. Train a GRU head on WakeHuBERT features, with background noise, music and room reverb
   mixed in during training.
4. Export ONNX: the batch head and a stateful streaming head.
5. Run a streaming detection demo on a wav file.

The counts below (300 positives, a few epochs) keep the run on a CPU. The published
WakeHuBERT results used about 900 positives with augmentation: a GRU head trained on 900
synthetic "alexa" clips detected 95% (95% interval 92–97) of the real speakers in the
Picovoice wake-word benchmark at a threshold set for 0.5 false activations per hour, in a
single run.

## 1 — Install

The `datagen` extra brings the TTS plugin manager with edge-tts, Hugging Face `datasets`
and the ONNX voice-activity detector used to trim silence. Every installed OVOS TTS plugin
is used for synthesis, so installing more (phoonnx, piper, google-tx) adds voice variety.

In [ ]:
%pip install -q --pre "wakeforge[datagen,torchcodec]"

## 2 — Configuration

Every setting can be overridden by an environment variable of the same name.

In [ ]:
import os
from pathlib import Path

WAKE_WORD   = os.environ.get("WAKE_WORD", "hey computer")
LANG        = os.environ.get("LANG_TTS", "en-us")
OUTPUT_DIR  = Path(os.environ.get("OUTPUT_DIR", "./nb12_wakehubert"))
N_POSITIVE  = int(os.environ.get("N_POSITIVE", "300"))    # ~900 with augmentation for the published results
MAX_NEGATIVE = int(os.environ.get("MAX_NEGATIVE", "600"))  # per negative source
FEATURIZER  = os.environ.get("FEATURIZER", "wakehubert")  # or "wakehubert-int8", or another published extractor
EPOCHS      = int(os.environ.get("EPOCHS", "8"))
BATCH_SIZE  = int(os.environ.get("BATCH_SIZE", "32"))
DEVICE      = os.environ.get("DEVICE", "cpu")
SEED        = 42

DATASET_DIR = OUTPUT_DIR / "dataset"
MODEL_DIR   = OUTPUT_DIR / "model"

## 3 — Synthetic positives, negatives and augmentation audio

`run_datagen_pipeline` is what `ww_trainer-quickstart` runs: it synthesises `N_POSITIVE`
utterances of the wake word with the installed TTS plugins, downloads general and speech
negatives from Hugging Face, adds grapheme-level near-miss negatives (`adversarial=True`),
downloads background noise, music and room impulse responses, and writes train/test
metadata CSVs.

In [ ]:
from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline

data = run_datagen_pipeline(DatagenConfig(
    wake_word=WAKE_WORD,
    output_dir=DATASET_DIR,
    n_positive=N_POSITIVE,
    max_negative=MAX_NEGATIVE,
    lang=LANG,
    adversarial=True,
    download_augmentation=True,
    seed=SEED,
))
print(f"positives: {data.n_positive}  negatives: {data.n_negative}")
print("train:", data.train_csv, " test:", data.test_csv)

## 4 — Load the featurizer

`OnnxFeatureExtractor.from_pretrained` downloads the ONNX and its `config.json` into the
shared Hugging Face cache. The feature size and hop come from that config; `context_samples`
is the 2.5 s of past audio that determines one frame.

In [ ]:
import torch
from ww_trainer.feats import OnnxFeatureExtractor

feat = OnnxFeatureExtractor.from_pretrained(FEATURIZER, device=DEVICE)
print("ONNX:", feat.model_path)
print(f"feature_dim={feat.feature_dim}  hop={feat.hop_samples}  "
      f"{feat.frame_rate_hz:.0f} fps  context={feat.context_samples} samples")
print("1.5 s of audio ->", tuple(feat(torch.zeros(1, 24000)).shape))

## 5 — Train a GRU head

`featurizer_type="wakehubert"` builds the same extractor inside the trainer, so no
`feature_dim` is passed. The noise, music and reverb folders from step 3 are mixed into the
training clips on the fly. The command-line equivalent is:

```bash
ww_trainer-train --wake-word "hey computer" --metadata <train.csv> --test-metadata <test.csv> \
    --tier wakehubert --bg-noise-folder <bg_noise> --music-folder <music> --rir-folder <rir> \
    --epochs 8 --save-best
```

In [ ]:
from ww_trainer.trainer import WakeWordTrainer
from ww_trainer.utils import read_dataset_csv

augment = {key: str(path) for key, path in (
    ("bg_noise_folder", data.bg_noise_dir),
    ("music_folder", data.music_dir),
    ("rir_folder", data.rir_dir),
) if path is not None and Path(path).exists()}
print("augmentation folders:", augment)

trainer = WakeWordTrainer(
    arch="gru",
    featurizer="",
    featurizer_type=FEATURIZER,
    hidden_dim=128,
    losses_cfg=[{"name": "bce", "weight": 1.0}],
    wake_word=WAKE_WORD,
    device=DEVICE,
    export_onnx=True,
    seed=SEED,
    **augment,
)
best_f1 = trainer.train(
    output_dir=MODEL_DIR,
    train_data=read_dataset_csv(data.train_csv),
    test_data=read_dataset_csv(data.test_csv),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=5e-4,
    save_best=True,
)
print(f"best F1 on the synthetic test split: {best_f1:.3f}")

## 6 — Export ONNX

Training already wrote `best_f1.onnx`, the batch head that scores a whole clip of features.
The streaming head carries the GRU state between calls, so each 0.1 s chunk costs the same
however long the stream runs. Its `window` is the number of frames it averages over: 75
frames is 1.5 s at 50 fps.

In [ ]:
trainer.load_checkpoint(MODEL_DIR / "best_f1.pt")
trainer.model.eval()
WINDOW = 75
head_onnx = MODEL_DIR / "best_f1.onnx"
stream_onnx = MODEL_DIR / "head_streaming.onnx"
trainer.model.classifier.export_streaming_onnx(str(stream_onnx), window=WINDOW)
print(head_onnx, head_onnx.exists())
print(stream_onnx, stream_onnx.exists())

## 7 — Streaming detection demo

The demo builds a stream from a negative clip, a held-out positive and another negative,
and feeds it in 0.1 s chunks (a multiple of the 320-sample hop). With each chunk the
streamer re-featurizes `context_samples` of past audio, so streamed frames equal the
offline ones. The offline score of the positive alone, from `OnnxWakeWordInferencer`, is
printed for comparison.

Replace `DEMO_WAV` with your own 16 kHz mono recording to try it on real speech.

In [ ]:
import numpy as np
import soundfile as sf
from ww_trainer.inference import OnnxStreamingWakeWord, OnnxWakeWordInferencer

test_rows = read_dataset_csv(data.test_csv)
pos_wav = next(p for p, label in test_rows if str(label) == "1")
neg_wav = next(p for p, label in test_rows if str(label) == "0")

def load16k(path):
    import librosa
    audio, _ = librosa.load(path, sr=16000, mono=True)
    return audio.astype(np.float32)

DEMO_WAV = OUTPUT_DIR / "demo_stream.wav"
neg = load16k(neg_wav)[:16000]
stream = np.concatenate([neg, load16k(pos_wav), neg, np.zeros(8000, np.float32)])
sf.write(DEMO_WAV, stream, 16000)

audio = load16k(DEMO_WAV)
sw = OnnxStreamingWakeWord.from_extractor(feat, str(stream_onnx), window=WINDOW, hidden_dim=128)
CHUNK = 1600
scores = [sw.push(audio[i:i + CHUNK]) for i in range(0, len(audio) - CHUNK + 1, CHUNK)]
THRESHOLD = 0.5
for i, s in enumerate(scores):
    mark = "  <-- wake word" if s >= THRESHOLD else ""
    print(f"{i * CHUNK / 16000:5.1f} s  {s:.3f}{mark}")

offline = OnnxWakeWordInferencer(feat.model_path, str(head_onnx), device="cpu")
print(f"offline score of the positive clip: {offline.infer(load16k(pos_wav)):.3f}")

## Next steps

- Choose the threshold from false activations per hour measured on long streams of
  speech and noise that do not contain the wake word, not from 0.5.
- Scale to about 900 positives, add voice conversion (`vc_refs_dir` in `DatagenConfig`) and
  more TTS plugins for voice variety.
- `FEATURIZER = "wakehubert-int8"` uses the int8 featurizer for low-power CPUs. The other
  published extractors (`docs/reference/extractors.md`) can be selected the same way for
  comparison; the streaming demo refuses the ones that cannot be streamed.